# 5106 — Portfolio Optimization Project (Group 1)

**Title:** Regularized Portfolio Construction with LASSO and Ridge  
**Course:** 5106  
**Group:** 1

This notebook implements the full workflow required by the project brief:

- Data cleaning for the *100 Portfolios Formed on Size × Operating Profitability (OP)* daily dataset  
- Rolling/sliding-window backtest (**126 trading-day lookback**):contentReference[oaicite:1]{index=1}  
- Portfolio strategies: Equally-Weighted (EW), Minimum-Variance (MinVar), LASSO, Ridge  
- Model selection via cross-validation with **TimeSeriesSplit**  
- Evaluation on **Validation-2** (2025-01-02 → 2025-06-30) and **Final Test** (2025-08-01 → 2025-08-30):contentReference[oaicite:2]{index=2}

Notes:
- Returns are in **percent** in the raw dataset → convert to **decimals**.:contentReference[oaicite:3]{index=3}  
- Short selling is allowed unless otherwise constrained.:contentReference[oaicite:5]{index=5}


In [1]:
# === Imports & configuration ===
import re
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Tuple, Dict
from dataclasses import dataclass, field

import matplotlib.pyplot as plt

from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", 120)
np.set_printoptions(suppress=True, precision=6)

# For reproducibility
RNG_SEED = 42
np.random.seed(RNG_SEED)


In [2]:
# === Helpers for (5) standardization, (6) leverage/bounds, (7) objectives, (10) metrics ===
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV

MAX_ITER = 10000

def _safe_std(a, axis=0, ddof=1, eps=1e-12):
    s = a.std(axis=axis, ddof=ddof)
    if np.isscalar(s):
        return max(float(s), eps)
    s = s.astype(float, copy=True)
    s[s == 0] = eps
    return s


def fit_lasso_ridge_standardized(R_window, N, w_ew, model="lasso",
                                 lasso_alphas=np.logspace(-5, 1, 50),
                                 ridge_alphas=np.logspace(-4, 3, 50),
                                 tscv=None):
    """
    Standardize X and y inside window (no leakage), fit LASSO/Ridge (no intercept),
    then rescale coefficients back to original units.
    """
    R = np.asarray(R_window)
    X = R @ N                 # (T x (K-1))
    y = R @ w_ew              # (T,)

    X_mean = X.mean(axis=0, keepdims=True)
    X_std  = _safe_std(X, axis=0, ddof=1)
    Xz     = (X - X_mean) / X_std

    y_mean = y.mean()
    y_std  = _safe_std(y, axis=0, ddof=1)
    yz     = (y - y_mean) / y_std

    if tscv is None:
        tscv = TimeSeriesSplit(n_splits=5)

    if model == "lasso":
        est = LassoCV(alphas=lasso_alphas, cv=tscv, fit_intercept=False,
                      max_iter=MAX_ITER, random_state=RNG_SEED).fit(Xz, yz)
    elif model == "ridge":
        est = RidgeCV(alphas=ridge_alphas, cv=tscv, fit_intercept=False).fit(Xz, yz)
    else:
        raise ValueError("model must be 'lasso' or 'ridge'")

    # back to original units
    beta_hat = (est.coef_ / X_std.ravel()) * y_std
    return beta_hat


# def fit_lasso_ridge_on_window(R_window, N, w_ew, model="lasso",
#                               lasso_alphas=np.logspace(-4, 1, 30),
#                               ridge_alphas=np.logspace(-4, 3, 30),
#                               tscv=None):
#     """(5) Standardize X/y within the window, fit Lasso/Ridge, rescale coefficients back."""
#     R = np.asarray(R_window)
#     X = R @ N                 # (T x (K-1))
#     y = R @ w_ew              # (T,)

#     X_mean = X.mean(axis=0, keepdims=True)
#     X_std  = _safe_std(X, axis=0, ddof=1)
#     Xz     = (X - X_mean) / X_std

#     y_mean = y.mean()
#     y_std  = _safe_std(y, axis=0, ddof=1)
#     yz     = (y - y_mean) / y_std

#     if model == "lasso":
#         est = LassoCV(alphas=lasso_alphas, cv=tscv, fit_intercept=False,
#                       max_iter=MAX_ITER, random_state=RNG_SEED).fit(Xz, yz)
#     elif model == "ridge":
#         est = RidgeCV(alphas=ridge_alphas, fit_intercept=False, cv=tscv).fit(Xz, yz)
#     else:
#         raise ValueError("model must be 'lasso' or 'ridge'")

#     beta_hat = (est.coef_ / X_std.ravel()) * y_std   # back to original units
#     return beta_hat

def clamp_gross(w, L=1.5):
    """ Cap gross leverage ||w||_1 ≤ L by proportional scaling."""
    gross = np.abs(w).sum()
    return w if gross <= L else (w * (L / gross))


def long_only_simplex(w):
    """(6) Project to simplex: w ≥ 0, sum w = 1."""
    w = np.clip(w, 0.0, None)
    s = w.sum()
    return w if s == 0 else (w / s)

def sharpe_annual(r_daily, trading_days=252, rf_daily=0.0):
    """(10) Annualized Sharpe using daily returns."""
    r = np.asarray(r_daily) - rf_daily
    mu = r.mean()
    sd = r.std(ddof=1)
    return np.sqrt(trading_days) * mu / sd if sd > 0 else np.nan


# (7A) Penalized minimum-variance (ridge on weights)
def minvar_ridge_weights(R_window, lam=1e-2, long_only=False):
    """
    Penalized min-variance with L2 on weights: argmin_w w' Σ w + lam * ||w||_2^2
    s.t. sum w = 1 (and optional long-only).
    Closed form for unconstrained sign; long-only is projected as a safe fallback.
    """
    R = np.asarray(R_window)
    Sigma = np.cov(R, rowvar=False)
    K = Sigma.shape[0]
    A = Sigma + lam * np.eye(K)
    ones = np.ones(K)
    x = np.linalg.solve(A, ones)
    w = x / (ones @ x)
    if long_only:
        w = long_only_simplex(w)
    return w

def minvar_ridge_cv_weights(R_window, lam_grid=np.logspace(-5, 0, 20),
                            n_splits=5, long_only=False):
    """
    Simple time-series CV to choose lam that minimizes validation realized variance
    (w_t' r_{t+1}) variance across folds.
    """
    R = np.asarray(R_window)
    T = R.shape[0]
    tscv = TimeSeriesSplit(n_splits=min(n_splits, max(2, T // 20)))

    best_lam, best_score = None, np.inf
    for lam in lam_grid:
        fold_vars = []
        for tr_idx, val_idx in tscv.split(R):
            if len(val_idx) == 0 or val_idx[-1] >= T - 1:
                continue
            R_tr, R_val = R[tr_idx], R[val_idx]
            w = minvar_ridge_weights(R_tr, lam=lam, long_only=long_only)
            # realized returns over validation days (use weights fixed from training)
            r_val = (R[val_idx + 1] @ w)  # next-day realized (shift by +1)
            if r_val.size > 1:
                fold_vars.append(np.var(r_val, ddof=1))
        if len(fold_vars):
            score = float(np.mean(fold_vars))
            if score < best_score:
                best_score, best_lam = score, lam

    if best_lam is None:
        best_lam = lam_grid[len(lam_grid)//2]
    return minvar_ridge_weights(R_window, lam=best_lam, long_only=long_only)

# (7B) Mean–variance (forecast-then-optimize)
def mean_variance_weights(mu, Sigma, risk_aversion=5.0, long_only=False, gross_cap=None):
    K = len(mu)
    ones = np.ones(K)
    x = np.linalg.solve(Sigma + 1e-6*np.eye(K), mu)
    w = x / (ones @ x)            # sum-to-1
    if long_only:
        w = long_only_simplex(w)
    if gross_cap is not None:
        w = clamp_gross(w, L=gross_cap)
    return w


def portfolio_returns_with_costs(R_panel, W_path, cost_per_dollar=0.0):
    """
    (10) Compute net portfolio returns after linear trading costs.
    - R_panel: (T x K) daily asset returns
    - W_path:  (T x K) weights used on each day (pre-trade weights for that day's return)
    - cost_per_dollar: cost * turnover_t, where turnover_t = sum(|w_t - w_{t-1}|)
    Returns r_net (T,), daily_turnover (T,)
    """
    R = np.asarray(R_panel)
    W = np.asarray(W_path)
    r_gross = (W * R).sum(axis=1)
    dW = np.vstack([np.zeros_like(W[0]), np.abs(W[1:] - W[:-1])])
    turnover = dW.sum(axis=1)
    r_net = r_gross - cost_per_dollar * turnover
    return r_net, turnover

In [3]:
import pandas as pd
import numpy as np

# --- Set the path to your cleaned CSV file ---
DATA_PATH = '100_Portfolios_ME_OP_10x10_daily.csv'

df = pd.read_csv(DATA_PATH, index_col=False)
    
# Standardize column names
df = df.rename(columns={df.columns[0]: "date"})

# # Convert date column to datetime
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")

# df = df.apply(pd.to_numeric, errors="coerce")
# df = df.replace({-99.99: np.nan, -999: np.nan})
    
df

/tmp/ipykernel_1234044/753080307.py:7: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, index_col=False)


,date,SMALL LoOP,ME1 OP2,ME1 OP3,ME1 OP4,ME1 OP5,ME1 OP6,ME1 OP7,ME1 OP8,ME1 OP9,SMALL HiOP,ME2 OP1,ME2 OP2,ME2 OP3,ME2 OP4,ME2 OP5,ME2 OP6,ME2 OP7,ME2 OP8,ME2 OP9,ME2 OP10,ME3 OP1,ME3 OP2,ME3 OP3,ME3 OP4,ME3 OP5,ME3 OP6,ME3 OP7,ME3 OP8,ME3 OP9,ME3 OP10,ME4 OP1,ME4 OP2,ME4 OP3,ME4 OP4,ME4 OP5,ME4 OP6,ME4 OP7,ME4 OP8,ME4 OP9,ME4 OP10,ME5 OP1,ME5 OP2,ME5 OP3,ME5 OP4,ME5 OP5,ME5 OP6,ME5 OP7,ME5 OP8,ME5 OP9,ME5 OP10,ME6 OP1,ME6 OP2,ME6 OP3,ME6 OP4,ME6 OP5,ME6 OP6,ME6 OP7,ME6 OP8,ME6 OP9,ME6 OP10,ME7 OP1,ME7 OP2,ME7 OP3,ME7 OP4,ME7 OP5,ME7 OP6,ME7 OP7,ME7 OP8,ME7 OP9,ME7 OP10,ME8 OP1,ME8 OP2,ME8 OP3,ME8 OP4,ME8 OP5,ME8 OP6,ME8 OP7,ME8 OP8,ME8 OP9,ME8 OP10,ME9 OP1,ME9 OP2,ME9 OP3,ME9 OP4,ME9 OP5,ME9 OP6,ME9 OP7,ME9 OP8,ME9 OP9,ME9 OP10,BIG LoOP,ME10 OP2,ME10 OP3,ME10 OP4,ME10 OP5,ME10 OP6,ME10 OP7,ME10 OP8,ME10 OP9,BIG HiOP
0,1963-07-01,-0.9,-0.65,-1.09,-1.2,-0.79,-0.36,-0.02,0.28,-0.92,-0.35,-0.31,-0.95,-1.35,-0.53,-0.97,-0.58,0.58,-0.33,-1.58,-0.5,-0.49,-0.76,-0.26,-1.01,-1.11,1.1,0.14,-0.7,-0.2,-1.3,-1.27,-0.7,-0.38,-1.64,-1.46,0.57,-0.63,-2.01,-1.4,-0.98,-0.82,-0.52,-0.39,-0.68,-1.65,-0.82,-0.83,-0.69,-0.48,-0.18,0.18,-0.01,-0.65,-0.61,0.41,-1.0,-0.55,-0.36,-0.67,-0.73,-0.34,-1.0,-0.06,-0.38,-0.03,-0.94,-0.38,-0.25,-0.55,-1.44,-0.47,-0.73,-0.13,-0.23,-0.55,-0.51,-0.68,-1.07,-0.24,-1.46,-2.17,-1.37,-0.75,-1.31,-0.58,-0.58,-0.29,-0.44,-0.57,-1.15,-0.91,-0.72,-0.58,-0.86,-0.56,-0.52,-0.65,-0.61,-0.56,-0.74
1,1963-07-02,0.26,0.18,0.24,-0.03,0.53,0.67,1.22,-0.12,-0.07,-0.11,0.5,0.38,1.9,0.79,1.27,1.79,1.1,0.67,0.74,-0.17,0.15,1.24,0.39,0.49,-0.23,0.08,0.29,0.82,-0.41,0.52,1.0,1.52,0.54,0.8,0.94,1.37,0.84,2.45,0.05,0.69,0.41,0.71,0.87,0.08,1.03,1.75,0.29,0.32,0.99,-0.36,0.12,0.59,0.74,-0.37,0.48,1.11,0.49,1.2,0.93,0.49,0.57,0.82,-0.13,0.72,0.46,0.9,0.54,0.36,0.21,1.01,0.7,-0.31,0.43,0.14,0.7,0.99,0.66,0.98,1.08,1.16,2.22,0.37,0.25,1.29,0.05,0.91,0.26,0.83,0.72,1.09,0.74,1.13,0.9,1.06,0.8,0.5,0.75,0.79,0.99,1.07
2,1963-07-03,0.71,0.79,-0.5,0.1,0.47,1.21,-0.06,0.34,-0.3,0.6,0.6,0.55,0.78,0.78,0.47,1.41,-2.0,0.96,0.65,0.52,-0.11,0.87,0.15,0.77,1.03,-0.3,-0.46,0.61,1.19,0.98,0.3,0.26,-0.28,0.35,0.34,0.41,1.27,-0.34,1.27,0.92,0.68,0.27,0.85,-0.26,0.48,1.04,-0.66,-0.11,0.81,0.89,0.89,0.82,0.34,0.67,0.56,-0.73,-0.09,0.97,0.2,1.46,0.7,0.14,0.62,0.86,0.46,0.79,0.26,0.32,0.93,0.03,-0.62,0.22,0.31,1.04,0.73,0.18,0.02,0.55,0.93,1.64,4.35,1.41,0.47,0.25,0.71,0.95,0.65,0.26,1.0,0.73,-0.24,0.13,0.77,1.28,0.83,0.22,0.47,1.02,0.36,0.3
3,1963-07-05,0.53,0.15,0.69,0.53,-0.07,-0.39,0.38,-0.14,0.03,0.51,1.07,0.01,1.18,0.92,0.0,1.04,-0.07,-0.04,0.23,1.05,0.74,-0.18,-0.42,0.46,-0.01,0.31,-0.02,0.2,0.56,1.06,0.9,1.19,-0.35,0.51,0.35,1.74,-0.34,3.74,0.42,-0.28,0.75,0.41,-0.37,0.65,0.44,0.84,0.38,1.42,0.43,0.48,1.63,0.59,0.01,-0.18,0.51,0.43,-0.62,0.01,-0.05,1.4,0.55,1.08,-0.05,-0.08,-0.27,0.46,0.32,0.74,0.12,1.07,0.79,0.95,0.01,0.33,-0.24,-0.05,-0.08,0.23,0.92,0.48,3.65,-0.12,0.56,0.12,0.76,0.33,0.48,0.35,0.1,0.31,0.28,0.8,0.18,0.97,0.19,0.04,0.72,0.08,0.38,0.76
4,1963-07-08,-0.39,-0.75,-0.29,-0.57,-1.49,-0.07,-0.91,-0.11,-1.13,0.07,-0.22,0.16,-0.77,0.46,0.55,-0.98,-0.4,0.51,-1.03,-0.47,-0.49,-0.52,0.38,-0.51,-0.52,-0.25,-0.3,-0.13,-0.57,-0.97,-0.92,-0.01,-0.47,-1.01,-1.09,-0.29,-0.79,-1.46,-1.3,-1.08,-0.64,-0.71,-0.22,-0.18,-1.05,-1.28,-0.45,-1.08,-1.16,-0.4,-0.61,0.18,-0.43,-0.86,-0.44,-0.46,-0.25,-0.8,-0.55,-0.09,-0.64,-0.76,-0.16,-0.23,-0.33,-0.01,-0.7,0.01,-0.34,0.01,-0.26,0.49,-0.03,-0.68,-0.25,-0.33,0.02,-0.63,-0.59,0.32,-1.01,-0.63,-0.9,-0.46,0.07,-0.42,-0.49,-0.3,-0.64,0.27,-0.22,-0.35,-0.64,-0.9,-0.73,-0.12,-0.26,-0.7,-0.63,-1.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62420,2025-06-26,10

In [4]:
TS_SPLITS = 5
MAX_ITER = 10000
RNG_SEED = 42

@dataclass
class ModelGrids:
    lasso_alphas: np.ndarray = None
    ridge_alphas: np.ndarray = None
    
    def __post_init__(self):
        if self.lasso_alphas is None:
            self.lasso_alphas = np.logspace(-5, 1, 50)
        if self.ridge_alphas is None:
            self.ridge_alphas = np.logspace(-3, 3, 50)

def ew_weights(p: int) -> np.ndarray:
    """Equal-weight portfolio weights."""
    return np.ones(p) / p

def build_N(p: int) -> np.ndarray:
    """Build the N matrix for the regression approach."""
    N = np.zeros((p, p-1))
    for j in range(p-1):
        N[j, j] = 1
        N[p-1, j] = -1
    return N


def minvar_weights(returns: pd.DataFrame, shrinkage: float = 0.1) -> np.ndarray:
    """
    Compute minimum variance portfolio weights with covariance shrinkage.
    """
    cov = returns.cov().values
    p = cov.shape[0]
    
    # Shrinkage toward diagonal
    diag_cov = np.diag(np.diag(cov))
    cov_shrunk = (1 - shrinkage) * cov + shrinkage * diag_cov
    
    try:
        inv_cov = np.linalg.inv(cov_shrunk)
        ones = np.ones(p)
        w = inv_cov @ ones
        w = w / w.sum()
    except np.linalg.LinAlgError:
        # If inversion fails, use equal weights
        w = ew_weights(p)
    
    return w


def fit_day_models(train_win: pd.DataFrame,
                   ret_t1: pd.Series,
                   grids: ModelGrids,
                   *,
                   long_only=False,
                   gross_cap=None,
                   use_minvar_ridge_cv=True) -> dict:
    """
    (5) Standardizes X/y inside the window for LASSO/Ridge; (6) clamps/bounds weights;
    (7) adds objective-aligned MinVar-Ridge (CV on lam); (10) returns one-step OOS.
    Inputs:
      - train_win: returns DataFrame for the lookback window (T x K)
      - ret_t1:    next day's returns (Series over same K columns)
    """
    cols_ok = train_win.columns
    p = len(cols_ok)
    R = train_win.values
    N = build_N(p)
    w_ew = ew_weights(p)

    # Time-series CV
    tscv = TimeSeriesSplit(n_splits=min(TS_SPLITS, max(2, len(train_win)//10)))

    # (5) LASSO/Ridge with per-window standardization + rescale back:
    beta_lasso = fit_lasso_ridge_standardized(R, N, w_ew,
                                              model="lasso",
                                              lasso_alphas=grids.lasso_alphas,
                                              tscv=tscv)
    w_lasso = w_ew - N @ beta_lasso

    beta_ridge = fit_lasso_ridge_standardized(R, N, w_ew,
                                              model="ridge",
                                              ridge_alphas=grids.ridge_alphas,
                                              tscv=tscv)
    w_ridge = w_ew - N @ beta_ridge

    # (7) Objective-aligned MinVar-Ridge (optional CV on lam)
    if use_minvar_ridge_cv:
        w_mv = minvar_ridge_cv_weights(R, long_only=long_only)
    else:
        w_mv = minvar_ridge_weights(R, lam=1e-2, long_only=long_only)

    # (6) Apply constraints
    def postprocess(w):
        if long_only:
            w = long_only_simplex(w)
        if gross_cap is not None:
            w = clamp_gross(w, L=gross_cap)
        # sanity: sum to 1
        s = w.sum()
        return w if np.isclose(s, 1.0) else (w / s)

    w_ew_pp    = postprocess(w_ew.copy())
    w_lasso_pp = postprocess(w_lasso.copy())
    w_ridge_pp = postprocess(w_ridge.copy())
    w_mv_pp    = postprocess(w_mv.copy())

    r_t1 = ret_t1.values
    return {
        "EW":    float(w_ew_pp @ r_t1),
        "MinVar":float(w_mv_pp @ r_t1),
        "LASSO": float(w_lasso_pp @ r_t1),
        "Ridge": float(w_ridge_pp @ r_t1),
        # Optionally return weights if you want to log/debug:
        # "_weights": {"EW": w_ew_pp, "MinVar": w_mv_pp, "LASSO": w_lasso_pp, "Ridge": w_ridge_pp, "cols": cols_ok}
    }



def backtest_strategies(
    returns: pd.DataFrame,
    start_date: str,
    end_date: str,
    lookback: int = 126,
    grids: ModelGrids = ModelGrids(),
    *,
    long_only=False,
    gross_cap=None,
    cost_per_dollar=0.0,   # (10) set to e.g. 0.0002 for 2 bps each dollar turned over
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    One-step-ahead backtest with (5) standardization, (6) leverage controls,
    (7) MinVar-Ridge, and (10) trading costs.
    Returns:
      oos_gross: daily gross returns for each strategy
      oos_net:   daily net returns after costs
    """
    returns = returns.copy().sort_index()

    # Ensure datetime index
    if not isinstance(returns.index, pd.DatetimeIndex):
        returns.index = pd.to_datetime(returns.index)

    start_ts, end_ts = pd.Timestamp(start_date), pd.Timestamp(end_date)
    mask_period = (returns.index >= start_ts) & (returns.index <= end_ts)
    dates_t = returns.index[mask_period]

    out = []
    idx = []
    W_store = {"EW": {}, "MinVar": {}, "LASSO": {}, "Ridge": {}}
    last_w = None

    for i, t in enumerate(dates_t):
        # window ends at t-1; predict t (next day) → index of t
        t_loc = returns.index.get_loc(t)
        if isinstance(t_loc, slice) or isinstance(t_loc, np.ndarray):
            t_loc = np.where(returns.index == t)[0][0]
        train_start_loc = t_loc - lookback - 1  # ensure last day in window is t-1
        train_end_loc   = t_loc - 1

        if train_start_loc < 0 or train_end_loc < 0:
            continue

        train_win = returns.iloc[train_start_loc:train_end_loc+1, :]
        # Drop assets that are entirely missing in window
        cols_ok = train_win.columns[~train_win.isna().all(axis=0)]
        train_win = train_win[cols_ok]

        if train_win.shape[1] < 2:
            continue

        # Next-day realized (no leakage): date t
        ret_t1 = returns.iloc[t_loc, :][cols_ok]

        try:
            daily = fit_day_models(
                train_win,
                ret_t1,
                grids,
                long_only=long_only,
                gross_cap=gross_cap,
                use_minvar_ridge_cv=True,
            )
        except Exception as e:
            print(f"Error on {t.date()}: {e}")
            continue

        out.append({k: daily[k] for k in ("EW","MinVar","LASSO","Ridge")})
        idx.append(t)

        # If you want costs, we need the weights used at t as well.
        # Recompute postprocessed weights consistent with fit_day_models:
        p = train_win.shape[1]
        N = build_N(p)
        w_ew = ew_weights(p)
        # Rebuild the three model weights (same routine as in fit_day_models)
        beta_lasso = fit_lasso_ridge_standardized(train_win.values, N, w_ew, model="lasso",
                                                  lasso_alphas=grids.lasso_alphas)
        w_lasso = w_ew - N @ beta_lasso
        beta_ridge = fit_lasso_ridge_standardized(train_win.values, N, w_ew, model="ridge",
                                                  ridge_alphas=grids.ridge_alphas)
        w_ridge = w_ew - N @ beta_ridge
        w_mv    = minvar_ridge_cv_weights(train_win.values, long_only=long_only)

        def postprocess(w):
            if long_only:
                w = long_only_simplex(w)
            if gross_cap is not None:
                w = clamp_gross(w, L=gross_cap)
            s = w.sum()
            return w if np.isclose(s, 1.0) else (w / s)

        w_map = {
            "EW":    postprocess(w_ew.copy()),
            "MinVar":postprocess(w_mv.copy()),
            "LASSO": postprocess(w_lasso.copy()),
            "Ridge": postprocess(w_ridge.copy()),
        }
        for k in W_store.keys():
            # align to full universe: here we just store on the reduced cols_ok set
            W_store[k][t] = pd.Series(w_map[k], index=cols_ok)

    if not len(out):
        return pd.DataFrame(), pd.DataFrame()

    oos_gross = pd.DataFrame(out, index=idx).sort_index()

    # DEBUG
    print("oos_gross shape:", oos_gross.shape)
    for k in W_store:
        print(k, "num dates stored:", len(W_store[k]))

      # (10) Compute net returns w/ costs per strategy — align by WEIGHT dates
    oos_net = {}
    for k in W_store.keys():
        # Build weights data frame: rows = dates we actually produced weights
        W_df = pd.DataFrame.from_dict(W_store[k], orient="index")  # index = dates
        W_df = W_df.sort_index()

        # Align returns to the exact same (date, asset) grid as W_df
        R_df = returns.reindex(index=W_df.index, columns=W_df.columns)

        # Safety fills (you can choose to drop instead, but fill keeps shapes equal)
        W_df = W_df.fillna(0.0)
        R_df = R_df.fillna(0.0)

        # Now shapes MUST match
        if R_df.shape != W_df.shape:
            raise ValueError(f"[internal] Alignment bug for {k}: R {R_df.shape} vs W {W_df.shape}")

        r_net, _turn = portfolio_returns_with_costs(
            R_df.values, W_df.values, cost_per_dollar=cost_per_dollar
        )
        oos_net[k] = pd.Series(r_net, index=W_df.index)

    # Stack all net returns and (optionally) align oos_gross to intersection of dates
    oos_net = pd.DataFrame(oos_net).sort_index()

    # Make sure gross and net share the same rows (dates you truly traded)
    common_idx = oos_gross.index.intersection(oos_net.index)
    oos_gross = oos_gross.reindex(common_idx)
    oos_net   = oos_net.reindex(common_idx)

    return oos_gross, oos_net



# Example usage with date validation
def validate_and_run_backtest(df, start_date, end_date, lookback=126):
    """
    Validate dates and run backtest with proper error handling.
    """
    # Check if df has a proper DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        print(f"Converting index from {type(df.index)} to DatetimeIndex...")
        df.index = pd.to_datetime(df.index)
    
    # Check date range
    print(f"Data date range: {df.index[0].date()} to {df.index[-1].date()}")
    
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    
    if start_ts > df.index[-1]:
        print(f"ERROR: Start date {start_date} is after the last date in data {df.index[-1].date()}")
        return None
    
    if end_ts < df.index[0]:
        print(f"ERROR: End date {end_date} is before the first date in data {df.index[0].date()}")
        return None
    
    # Adjust dates if needed
    actual_start = max(start_ts, df.index[0])
    actual_end = min(end_ts, df.index[-1])
    
    if actual_start != start_ts or actual_end != end_ts:
        print(f"Adjusted date range: {actual_start.date()} to {actual_end.date()}")
    
    grids = ModelGrids()
    return backtest_strategies(df, str(actual_start.date()), str(actual_end.date()), 
                              lookback=lookback, grids=grids)

In [5]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).set_index("date").sort_index()

# Now only the returns columns are left as data → safe to coerce
df = df.apply(pd.to_numeric, errors="coerce")
df = df/100.0
df = df.dropna(how="any")   # optional: drop rows with any missing values



In [6]:
def evaluate_oos(oos_returns: pd.DataFrame, title: str = "", ann_factor: int = 252):
    sr = {col: sharpe_annual(oos_returns[col], trading_days=ann_factor) for col in oos_returns.columns}
    sr_df = pd.DataFrame({"Sharpe_annual": sr})
    print(f"\n== Annualized Sharpe — {title} ==")
    display(sr_df.T)

    # Cumulative returns plot
    cum = (1.0 + oos_returns.fillna(0)).cumprod()
    plt.figure(figsize=(9, 5))
    cum.plot(ax=plt.gca())
    plt.title(f"Cumulative Returns — {title}")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.grid(True)
    plt.show()
    return sr_df

In [7]:
# === Backtest & Evaluate (updated to use new API and correct dates) ===
LOOKBACK = 126  # from the brief

VAL2_START = "2025-01-02"              # Validation-2 window
VAL2_END   = "2025-06-30"

TEST_START = "2025-08-01"
TEST_END   = "2025-08-30"

grids = ModelGrids()

# --- Run Validation-2 ---
val2_gross, val2_net = backtest_strategies(
    df, VAL2_START, VAL2_END,
    lookback=LOOKBACK, grids=grids,
    long_only=False,         # set True if you want no shorting
    gross_cap=1.5,           # cap ||w||_1 ≤ 1.5; adjust or set None
    cost_per_dollar=0.0002,  # (10) linear cost per $ turned (e.g., 2 bps)
)

print("Validation-2 shapes (gross, net):", val2_gross.shape, val2_net.shape)
display(val2_gross.head(3))

# Evaluate (annualized Sharpe) + cumulative growth
sr_val2_gross = evaluate_oos(val2_gross, title="Validation-2 (Gross)")
sr_val2_net   = evaluate_oos(val2_net,   title="Validation-2 (Net, after costs)")

# --- Run Final Test ---
test_gross, test_net = backtest_strategies(
    df, TEST_START, TEST_END,
    lookback=LOOKBACK, grids=grids,
    long_only=False,
    gross_cap=1.5,
    cost_per_dollar=0.0002,
)

print("Final Test shapes (gross, net):", test_gross.shape, test_net.shape)
display(test_gross.head(3))

sr_test_gross = evaluate_oos(test_gross, title="Final Test (Gross)")
sr_test_net   = evaluate_oos(test_net,   title="Final Test (Net, after costs)")


oos_gross shape: (488, 4)
EW num dates stored: 122
MinVar num dates stored: 122
LASSO num dates stored: 122
Ridge num dates stored: 122


ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
def backtest_with_weights(
    returns: pd.DataFrame,
    start_date: str,
    end_date: str,
    lookback: int = 126,
):
    returns = returns.copy().sort_index()
    # Enforce unique index/columns; required for safe reindex/loc
    if returns.index.has_duplicates:
        returns = returns[~returns.index.duplicated(keep="last")]
    if returns.columns.duplicated().any():
        returns = returns.loc[:, ~returns.columns.duplicated()]

    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)

    mask_period = (returns.index >= start_ts) & (returns.index <= end_ts)
    dates_t = returns.index[mask_period]
    if len(dates_t) == 0:
        raise ValueError("No dates inside the requested evaluation period.")

    oos_list = []
    idx = []
    weights = {"LASSO": [], "Ridge": []}
    weight_cols = None

    for t in dates_t:
        train_end_loc = returns.index.get_loc(t)
        if isinstance(train_end_loc, slice) or isinstance(train_end_loc, np.ndarray):
            train_end_loc = np.where(returns.index == t)[0][0]

        train_start_loc = train_end_loc - lookback
        if train_start_loc < 0:
            continue

        train_win = returns.iloc[train_start_loc:train_end_loc, :]

        # i used to have ret_t     = returns.iloc[train_end_loc, :], this created data leak ;(
        ret_t1_loc = train_end_loc + 1
        if ret_t1_loc >= len(returns):
            break  # or continue; no next-day data
        ret_t = returns.iloc[ret_t1_loc, cols_ok]

        cols_ok = train_win.columns[~train_win.isna().all(axis=0)]
        train_win = train_win[cols_ok]
        ret_t = ret_t[cols_ok]

        if train_win.shape[1] < 2:
            continue

        # Save weight columns once
        if weight_cols is None:
            weight_cols = cols_ok

        # Compute LASSO/Ridge weights (same as fit_day_models, but keep them)
        p = train_win.shape[1]
        R = train_win.values
        N = build_N(p)
        w_ew = ew_weights(p)
        y = R @ w_ew
        X = R @ N

        tscv = TimeSeriesSplit(n_splits=min(TS_SPLITS, max(2, len(train_win)//10)))

        lasso = LassoCV(alphas=np.logspace(-8, 8, 41), cv=tscv, fit_intercept=True,
                        max_iter=20000, random_state=RNG_SEED)
        lasso.fit(X, y)
        beta_lasso = lasso.coef_
        w_lasso = w_ew - N @ beta_lasso
        w_lasso /= w_lasso.sum()

        ridge = RidgeCV(alphas=np.logspace(-8, 8, 41), cv=tscv, fit_intercept=True)
        ridge.fit(X, y)
        beta_ridge = ridge.coef_
        w_ridge = w_ew - N @ beta_ridge
        w_ridge /= w_ridge.sum()

        w_mv = minvar_weights(train_win)
        r_t = ret_t.values

        oos_list.append({
            "EW": float(w_ew @ r_t),
            "MinVar": float(w_mv @ r_t),
            "LASSO": float(w_lasso @ r_t),
            "Ridge": float(w_ridge @ r_t),
        })
        idx.append(t)

        # Record full weight vectors aligned to columns
        w_l_series = pd.Series(np.nan, index=weight_cols)
        w_r_series = pd.Series(np.nan, index=weight_cols)
        w_l_series.loc[cols_ok] = w_lasso
        w_r_series.loc[cols_ok] = w_ridge
        weights["LASSO"].append(w_l_series)
        weights["Ridge"].append(w_r_series)

    oos = pd.DataFrame(oos_list, index=idx).sort_index()
    w_lasso_df = pd.DataFrame(weights["LASSO"], index=oos.index)
    w_ridge_df = pd.DataFrame(weights["Ridge"], index=oos.index)
    return oos, {"LASSO": w_lasso_df, "Ridge": w_ridge_df}

val2_oos_w, w_dict = backtest_with_weights(df, VAL2_START, VAL2_END, lookback=LOOKBACK)

# Weight volatility measure across the window
lasso_w_std = w_dict["LASSO"].std(skipna=True)
ridge_w_std = w_dict["Ridge"].std(skipna=True)

print("Top 10 most variable LASSO weights:")
display(lasso_w_std.sort_values(ascending=False).head(10).to_frame("std"))

print("Top 10 most variable Ridge weights:")
display(ridge_w_std.sort_values(ascending=False).head(10).to_frame("std"))


In [ ]:
DEBUG_DATE = "2025-01-02"
FIRST_P = 6

if pd.Timestamp(DEBUG_DATE) in df.index:
    loc_val = df.index.get_loc(pd.Timestamp(DEBUG_DATE))
    if isinstance(loc_val, slice):
        train_end = loc_val.start
    elif isinstance(loc_val, np.ndarray):
        train_end = loc_val[0]
    else:
        train_end = loc_val

    train_start = train_end - 126
    if train_start >= 0:
        train_win_6 = df.iloc[train_start:train_end, :FIRST_P]
        mv_w_6 = minvar_weights(train_win_6)
        print("MinVar weights (first 6 assets) at 2025-01-02:")
        print(np.round(mv_w_6, 3))
        r_t = df.iloc[train_end, :FIRST_P].values
        print("One-day OOS return (decimal):", float(mv_w_6 @ r_t))
    else:
        print("Insufficient history for the 126-day lookback on the debug date.")
else:
    print("Debug date 2025-01-02 not in the loaded dataset index.")


MinVar weights (first 6 assets) at 2025-01-02:
[-0.173  1.201  0.195  0.12  -0.088 -0.255]
One-day OOS return (decimal): 0.5607854130307486
